# Analyse des avis et alertes ANSSI

Notebook **autonome** : les fonctions du pipeline sont définies dans la cellule ci-dessous, il n'y a donc aucun fichier `.py` externe à placer à côté.

In [ ]:
# =============================================================
# MODULES DU PIPELINE (inlinés — aucun fichier .py externe requis)
# =============================================================
# Contenu de get_bulletin.py, cve_extraction.py, dataframe.py et
# get_cve_score.py regroupé ici pour rendre le notebook autonome.

# --- Imports communs ---
import feedparser
import requests
import re
import time
import json
from datetime import datetime
from pathlib import Path
import pandas as pd


# ===================== get_bulletin.py =====================
import feedparser
from datetime import datetime


def get_bulletin(url, limit=10):
    """
    Renvoie une liste de tout les bulletins de l'ANSSI dans un format utilisable pour le DataFrame.
    """
    feed = feedparser.parse(url)
    bulletins = []
    for entry in feed.entries[:limit]:
        lien = entry.link

        if "alerte" in lien:
            type_b = "alerte"
            dossier = "alertes"
        else:
            type_b = "avis"
            dossier = "avis"
        
        date_str = str(entry.get("published", "")[:16])  # Format Mon, 01 Jan 2024 
        parsed_date = datetime.strptime(date_str, "%a, %d %b %Y")
        date = parsed_date.strftime("%Y-%m-%d")

        bulletins.append({
            "id": entry.id.split("/")[-2], #La fin de l'URL c'est l'ID
            "titre": entry.title,
            "type": type_b,
            "date_publication": date,
            "lien": lien,
            "mode": "remote",
            "dossier": dossier
        })
    return bulletins


# ===================== cve_extraction.py =====================
import json
import re
import time
import requests
from pathlib import Path

# Répertoire de base du projet (à adapter selon l'arborescence réelle)
BASE_DIR = Path.cwd()  # adapte pour notebook (pas de __file__)

def extraire_cves(bulletin_id_ou_url, mode="local", type_bulletin="alertes"):
    """
    Extrait les CVEs depuis un bulletin ANSSI en mode local ou remote.

    Args:
        bulletin_id_ou_url (str): Nom de fichier (mode local) ou URL ANSSI (mode remote).
        mode (str): "local" pour lire un fichier, "remote" pour interroger l'API ANSSI.
        type_bulletin (str): Sous-dossier cible ("alertes", "avis", etc.).

    Returns:
        list[str]: Liste dédupliquée et triée de CVEs (ex. ["CVE-2024-1234", ...]).
    """

    contenu_brut = ""
    if mode == "local":
        chemin_fichier = BASE_DIR / type_bulletin / bulletin_id_ou_url

        try:
            with chemin_fichier.open("r", encoding="utf-8") as f:
                contenu_brut = f.read()
        except FileNotFoundError:
            print(f"[ERREUR] Fichier introuvable : {chemin_fichier}")
            return []
        except OSError as e:
            print(f"[ERREUR] Impossible de lire le fichier : {e}")
            return []

    elif mode == "remote":
        url_base = bulletin_id_ou_url.rstrip("/")
        url_json = f"{url_base}/json/"
        time.sleep(2)

        try:
            reponse = requests.get(url_json, timeout=15)
            reponse.raise_for_status()
            contenu_brut = reponse.text
        except requests.exceptions.HTTPError as e:
            print(f"[ERREUR] Réponse HTTP invalide : {e}")
            return []
        except requests.exceptions.ConnectionError as e:
            print(f"[ERREUR] Échec de connexion : {e}")
            return []
        except requests.exceptions.Timeout:
            print(f"[ERREUR] Délai d'attente dépassé pour : {url_json}")
            return []
        except requests.exceptions.RequestException as e:
            print(f"[ERREUR] Erreur réseau inattendue : {e}")
            return []

    else:
        print(f"[ERREUR] Mode inconnu : '{mode}'. Valeurs acceptées : 'local', 'remote'.")
        return []

    if not contenu_brut.strip():
        print("[AVERTISSEMENT] Le contenu récupéré est vide.")
        return []

    cves_collectees = set()
    try:
        donnees_json = json.loads(contenu_brut)
        liste_cves = donnees_json.get("cves", [])

        if isinstance(liste_cves, list):
            for entree in liste_cves:
                if isinstance(entree, dict):
                    for cle in ("name", "cve", "id", "cve_id"):
                        valeur = entree.get(cle, "")
                        if isinstance(valeur, str) and re.match(r"CVE-\d{4}-\d{4,7}", valeur):
                            cves_collectees.add(valeur.strip())
                elif isinstance(entree, str) and re.match(r"CVE-\d{4}-\d{4,7}", entree):
                    cves_collectees.add(entree.strip())

    except (json.JSONDecodeError, AttributeError, TypeError) as e:
        print(f"[INFO] Méthode 1 (parsing JSON) échouée, bascule sur regex : {e}")

    try:
        pattern_cve = re.compile(r"CVE-\d{4}-\d{4,7}")
        cves_regex = pattern_cve.findall(contenu_brut)
        cves_collectees.update(cves_regex)

    except (re.error, TypeError) as e:
        print(f"[ERREUR] Méthode 2 (regex) échouée : {e}")
    liste_finale = sorted(cves_collectees)

    if not liste_finale:
        print("[INFO] Aucune CVE trouvée dans ce bulletin.")

    return liste_finale


# Test de la fonction


# ===================== get_cve_score.py =====================
import requests
import time


def get_cve_scores(cve_id):
    """
    Récupère les scores CVSS, CWE (MITRE) et EPSS (FIRST) pour un CVE donné.
    Retourne un dictionnaire compatible avec le module de consolidation.
    """
    print(f"    -> [API MITRE] Interrogation pour {cve_id}...")
    time.sleep(2)  # Pause obligatoire de 2s pour soulager l'API

    url_mitre = f"https://cveawg.mitre.org/api/cve/{cve_id}"
    try:
        response = requests.get(url_mitre)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"      [ERREUR] MITRE API pour {cve_id} : {e}")
        return None

    # 1. Extraction Description
    try:
        description = data["containers"]["cna"]["descriptions"][0]["value"]
    except (KeyError, IndexError):
        description = "Non disponible"

    # 2. Extraction CVSS et Sévérité
    cvss_score = None
    base_severity = "Non renseigné"
    try:
        metrics = data["containers"]["cna"]["metrics"][0]
        if "cvssV3_1" in metrics:
            cvss_score = metrics["cvssV3_1"]["baseScore"]
            base_severity = metrics["cvssV3_1"].get("baseSeverity", "Non renseigné")
        elif "cvssV3_0" in metrics:
            cvss_score = metrics["cvssV3_0"]["baseScore"]
            base_severity = metrics["cvssV3_0"].get("baseSeverity", "Non renseigné")
    except (KeyError, IndexError):
        pass

    # 3. Extraction CWE
    cwe = "Non disponible"
    cwe_desc = "Non disponible"
    problemtype = data["containers"]["cna"].get("problemTypes", [{}])
    if problemtype and "descriptions" in problemtype[0]:
        try:
            cwe = problemtype[0]["descriptions"][0].get("cweId", "Non disponible")
            cwe_desc = problemtype[0]["descriptions"][0].get("description", "Non disponible")
        except (KeyError, IndexError):
            pass

    # 4. Extraction des produits (Correction de la boucle)
    produits_affectes_liste = []
    try:
        affected = data["containers"]["cna"]["affected"]
        for product in affected:
            vendor = product.get("vendor", "Non disponible")
            product_name = product.get("product", "Non disponible")
            versions = [v.get("version", "?") for v in product.get("versions", []) if v.get("status") == "affected"]

            produits_affectes_liste.append({
                "vendor": vendor,
                "produit": product_name,
                "versions": ", ".join(versions)
            })
    except KeyError:
        pass

    # 5. Extraction EPSS (FIRST)
    print(f"    -> [API FIRST] Interrogation EPSS pour {cve_id}...")
    time.sleep(2)  # Pause de 2s
    url_epss = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    epss_score = None
    try:
        response_epss = requests.get(url_epss)
        response_epss.raise_for_status()
        data_epss = response_epss.json()
        epss_data = data_epss.get("data", [])
        if epss_data:
            epss_score = epss_data[0]["epss"]
    except Exception as e:
        print(f"      [ERREUR] FIRST API pour {cve_id} : {e}")

    # 6. Retour structuré pour le DataFrame
    return {
        "identifiant": cve_id,
        "score_cvss": cvss_score,
        "base_severity": base_severity,
        "type_cwe": cwe,
        "score_epss": epss_score,
        "description": description,
        "produits_affectes": produits_affectes_liste
    }


# ===================== dataframe.py =====================
import pandas as pd
def consolider_en_dataframe(donnees_globales, nom_fichier_csv="donnees_enrichies.csv"):
    """
    Aplatit la structure imbriquée (Bulletins > CVEs > Produits) en un DataFrame.
    Règle d'or : 1 ligne = 1 combinaison unique (Bulletin + CVE + Produit).

    Args:
        donnees_globales (list): Liste de dictionnaires représentant les bulletins ANSSI enrichis.
        nom_fichier_csv (str): Chemin/nom du fichier CSV d'export. Défaut : "donnees_enrichies.csv".

    Returns:
        pd.DataFrame: DataFrame aplati contenant toutes les combinaisons Bulletin × CVE × Produit.
    """

    lignes_aplaties = []
    # BOUCLE 1 — Itération sur chaque bulletin ANSSI
    for bulletin in donnees_globales:

        # Extraction des champs de niveau "Bulletin" avec valeur par défaut
        id_anssi       = bulletin.get("id",               "Non renseigné")
        titre_anssi    = bulletin.get("titre",            "Non renseigné")
        type_bulletin  = bulletin.get("type",             "Non renseigné")
        date_pub       = bulletin.get("date_publication", "Non renseigné")
        lien_bulletin  = bulletin.get("lien",             "Non renseigné")

        # Récupération de la liste des CVEs enrichis
        cves_enrichis = bulletin.get("cves_enrichis", [])

        # CAS DÉGÉNÉRÉ 1 : Bulletin sans aucun CVE associé
        # → On crée quand même une ligne pour ne pas perdre le bulletin.
        if not cves_enrichis:
            lignes_aplaties.append({
                "ID ANSSI":           id_anssi,
                "Titre ANSSI":        titre_anssi,
                "Type":               type_bulletin,
                "Date de publication": date_pub,
                "Lien":               lien_bulletin,
                "Identifiant CVE":    "N/A",
                "Score CVSS":         "N/A",
                "Base Severity":      "N/A",
                "Type CWE":           "N/A",
                "Score EPSS":         "N/A",
                "Description":        "N/A",
                "Éditeur (Vendor)":   "N/A",
                "Produit":            "N/A",
                "Versions affectées": "N/A",
            })
            continue

        # BOUCLE 2 — Itération sur chaque CVE du bulletin courant
        for cve in cves_enrichis:
            identifiant_cve = cve.get("identifiant",    "Non renseigné")
            score_cvss      = cve.get("score_cvss",     None)
            base_severity   = cve.get("base_severity",  "Non renseigné")
            type_cwe        = cve.get("type_cwe",       "Non renseigné")
            score_epss      = cve.get("score_epss",     None)
            description     = cve.get("description",   "Non renseigné")
            produits_affectes = cve.get("produits_affectes", [])

            # CAS DÉGÉNÉRÉ 2 : CVE sans aucun produit associé
            # → On crée une ligne avec les infos Bulletin + CVE, sans produit.
            if not produits_affectes:
                lignes_aplaties.append({
                    "ID ANSSI":           id_anssi,
                    "Titre ANSSI":        titre_anssi,
                    "Type":               type_bulletin,
                    "Date de publication": date_pub,
                    "Lien":               lien_bulletin,
                    "Identifiant CVE":    identifiant_cve,
                    "Score CVSS":         score_cvss,
                    "Base Severity":      base_severity,
                    "Type CWE":           type_cwe,
                    "Score EPSS":         score_epss,
                    "Description":        description,
                    "Éditeur (Vendor)":   None,
                    "Produit":            None,
                    "Versions affectées": None,
                })
                continue

            # BOUCLE 3 — Itération sur chaque produit affecté par le CVE courant
            for produit in produits_affectes:
                editeur          = produit.get("vendor",   "Non renseigné")
                nom_produit      = produit.get("produit",  "Non renseigné")
                versions         = produit.get("versions", "Non renseigné")
                ligne = {
                    "ID ANSSI":            id_anssi,
                    "Titre ANSSI":         titre_anssi.replace("\n", " "),
                    "Type":                type_bulletin.replace("\n", " "),
                    "Date de publication": date_pub,
                    "Lien":                lien_bulletin,
                    "Identifiant CVE":     identifiant_cve.replace("\n", " "),
                    "Score CVSS":          score_cvss,
                    "Base Severity":       base_severity,
                    "Type CWE":            type_cwe.replace("\n", " "),
                    "Score EPSS":          score_epss,
                    "Description":         description.replace("\n", " "),
                    "Éditeur (Vendor)":    editeur.replace("\n", " "),
                    "Produit":             nom_produit.replace("\n", " "),
                    "Versions affectées":  versions.replace("\n", " "),
                }
                lignes_aplaties.append(ligne)

    # CONSTRUCTION DU DATAFRAME — une seule passe, hors de toute boucle.
    df = pd.DataFrame(lignes_aplaties)

    # EXPORT CSV
    # - index=False  : on n'exporte pas l'index pandas
    # - encoding='utf-8' : compatibilité maximale pour les caractères spéciaux
    df.to_csv(nom_fichier_csv, index=False, encoding="utf-8")
    return df

print("Modules du pipeline chargés (get_bulletin, extraire_cves, get_cve_scores, consolider_en_dataframe).")


## Choix de la source des données

Le notebook fonctionne en deux modes, sélectionnés par la variable `MODE` :
- **`remote`** : interroge les flux RSS et API en ligne de l'ANSSI.
- **`local`** : lit les fichiers JSON pré-téléchargés dans `./alertes/` et `./avis/` (cf. consigne, section 8).

Toute la suite (cache, enrichissement, consolidation, visualisations) est identique.

In [ ]:
# =============================================================
# CELLULE MODE — Choix de la source des données
# =============================================================
# MODE = "remote" : on interroge les flux RSS + API en ligne de l'ANSSI.
# MODE = "local"  : on lit les fichiers JSON pré-téléchargés dans les
#                   dossiers ./alertes/ et ./avis/ (cf. consigne, section 8).
#
# Dans les deux cas, la suite du notebook (cache, enrichissement,
# consolidation, visualisations) est IDENTIQUE.

MODE = "remote"          # <-- mettre "local" pour travailler hors-ligne
LIMIT = 10               # nb max de bulletins par flux (mode remote)

assert MODE in ("remote", "local"), "MODE doit valoir 'remote' ou 'local'."
print(f"MODE sélectionné : {MODE.upper()}")

In [ ]:
# =============================================================
# CELLULE MODE-LOCAL — Lecture des bulletins depuis les dossiers locaux
# =============================================================
# get_bulletin (du module) ne gère que le flux RSS (remote). En local, il n'y
# a pas de RSS : on liste les fichiers de ./alertes/ et ./avis/ et on lit les
# métadonnées dans chaque JSON. Cette fonction produit la MÊME structure que
# get_bulletin pour que la suite ne change pas.
import os
import json
from pathlib import Path

def get_bulletin_local(dossiers=("alertes", "avis")):
    """Construit la liste des bulletins à partir des fichiers JSON locaux."""
    bulletins = []
    for dossier in dossiers:
        chemin = Path(dossier)
        if not chemin.is_dir():
            print(f"[AVERTISSEMENT] Dossier local absent : {chemin}/")
            continue

        type_b = "alerte" if dossier.startswith("alerte") else "avis"

        for nom_fichier in os.listdir(chemin):
            fichier = chemin / nom_fichier
            if not fichier.is_file():
                continue
            try:
                with fichier.open("r", encoding="utf-8") as f:
                    data = json.load(f)
            except (json.JSONDecodeError, OSError) as e:
                print(f"[ERREUR] Lecture {fichier} : {e}")
                continue

            # Métadonnées : on tente plusieurs clés (le format ANSSI peut varier)
            titre = data.get("title", nom_fichier)
            ref   = data.get("reference", nom_fichier)

            # Date : on cherche dans 'revisions' puis quelques clés alternatives
            date_pub = "Non renseigné"
            revisions = data.get("revisions", [])
            if revisions and isinstance(revisions, list):
                date_pub = (revisions[0].get("revision_date")
                            or revisions[0].get("date") or date_pub)
            else:
                date_pub = data.get("closed_at") or data.get("last_revision") or date_pub
            if isinstance(date_pub, str):
                date_pub = date_pub[:10]   # garde YYYY-MM-DD si format long

            bulletins.append({
                "id": ref,
                "titre": titre,
                "type": type_b,
                "date_publication": date_pub,
                "lien": f"https://www.cert.ssi.gouv.fr/{type_b}/{ref}/",
                "mode": "local",
                "dossier": dossier,
            })
    return bulletins


In [13]:
import feedparser 
url = "https://www.cert.ssi.gouv.fr/feed/" 
rss_feed = feedparser.parse(url) 
entries_link = []
for entry in rss_feed.entries: 
    print("Titre :", entry.title) 
    print("Description:", entry.description) 
    print("Lien :", entry.link) 
    entries_link.append(entry.link)
    print("Date :", entry.published)

Titre : [Màj] Vulnérabilité dans Microsoft Exchange Server (15 mai 2026)
Description: [Mise à jour du 11 juin 2026] Le 9 juin 2026, Microsoft a publié des versions correctives. [Publication initiale] Le 14 mai 2026, Microsoft a publié un avis de sécurité concernant la vulnérabilité CVE-2026-42897 affectant Exchange Server. Elle permet à un attaquant non authentifié de provoquer...
Lien : https://www.cert.ssi.gouv.fr/alerte/CERTFR-2026-ALE-005/
Date : Fri, 15 May 2026 00:00:00 +0000
Titre : Vulnérabilité dans Cisco Catalyst SD-WAN (05 juin 2026)
Description: Une vulnérabilité a été découverte dans Cisco Catalyst SD-WAN. Elle permet à un attaquant de provoquer une élévation de privilèges. Cisco indique que la vulnérabilité CVE-2026-20245 est activement exploitée.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0699/
Date : Fri, 05 Jun 2026 00:00:00 +0000
Titre : Multiples vulnérabilités dans les produits Fortinet (10 juin 2026)
Description: De multiples vulnérabilités ont été dé

In [14]:
import requests 
import re 
cve_ids = []
for link in entries_link:
    url = link + "json/"
    
    try:
        response = requests.get(url)
        # Lève une erreur si le code HTTP n'est pas 200 (ex: 404, 500)
        response.raise_for_status() 
        
        # Tente de décoder le JSON
        data = response.json() 
        
    except requests.exceptions.RequestException as e:
        # Attrape les erreurs de connexion, timeout, 404, etc.
        print(f"Erreur de connexion ou HTTP pour {url} : {e}")
        continue # Passe au lien suivant
        
    except requests.exceptions.JSONDecodeError:
        # Attrape le problème que tu avais (réponse non-JSON)
        print(f"Réponse JSON invalide pour {url}. Le site a renvoyé du texte brut ou du HTML.")
        # Optionnel : décommenter la ligne suivante pour inspecter le problème en direct
        # print("Contenu reçu :", response.text[:200]) 
        continue # Passe au lien suivant

    # --- Tout ce qui est en dessous ne s'exécute que si le try a réussi ---
    
    # Extraction des CVE références dans la clé cves du dict data 
    # (Sécurisé avec .get() au cas où la clé "cves" serait absente)
    cves_data = data.get("cves", [])
    print("CVE référencés :", cves_data) 

    
    #Si on a des CVE, on extrait proprement les identifiants 'name'
    if len(cves_data) > 0:
        for cve in cves_data:
            if "name" in cve.keys():
                cve_ids.append(cve["name"])
    else:
        print("Aucun CVE référencé trouvé pour ce lien.")
    
    #Extraction des CVE avec une regex (ton code actuel)
    cve_pattern = r"CVE-\d{4}-\d{4,7}" 
    cve_list = list(set(re.findall(cve_pattern, str(data)))) 
    print("CVE trouvés :", cve_list)
    print("-" * 40)

CVE référencés : [{'name': 'CVE-2026-42897', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-42897'}]
CVE trouvés : ['CVE-2026-42897']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-20245', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-20245'}]
CVE trouvés : ['CVE-2026-20245']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-25089', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-25089'}, {'name': 'CVE-2025-67862', 'url': 'https://www.cve.org/CVERecord?id=CVE-2025-67862'}, {'name': 'CVE-2026-49938', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-49938'}]
CVE trouvés : ['CVE-2025-67862', 'CVE-2026-49938', 'CVE-2026-25089']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-45257', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-45257'}, {'name': 'CVE-2026-49413', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-49413'}]
CVE trouvés : ['CVE-2026-49413', 'CVE-2026-45257']
-----------------

## Partie 4 — Consolidation des données

Les cellules ci-dessus (0–2) montrent l'extraction RSS et l'identification des CVE. Pour la consolidation, on s'appuie sur les **modules du pipeline** (`get_bulletin`, `cve_extraction`, `dataframe`) afin de conserver le lien CVE → bulletin, et on ajoute un **cache disque** pour ne jamais réinterroger un CVE déjà vu (gros gain sur les bulletins à 200+ CVE). Le résultat est exporté en `donnees_enrichies.csv`.

### 4A. Extraction des bulletins et de leurs CVE (via modules)

In [ ]:
# =============================================================
# CELLULE 4A — Extraction des bulletins et de leurs CVE
# =============================================================
# Selon MODE (défini plus haut) :
#   - "remote" : flux RSS en ligne via get_bulletin()
#   - "local"  : fichiers JSON locaux via get_bulletin_local()
# Dans les deux cas, on obtient la même structure de bulletins.

FEED_ALERTES = "https://www.cert.ssi.gouv.fr/alerte/feed/"
FEED_AVIS    = "https://www.cert.ssi.gouv.fr/avis/feed/"

if MODE == "remote":
    bulletins_meta = get_bulletin(FEED_ALERTES, limit=LIMIT) + get_bulletin(FEED_AVIS, limit=LIMIT)
else:  # local
    bulletins_meta = get_bulletin_local(dossiers=("alertes", "avis"))

print(f"{len(bulletins_meta)} bulletins récupérés (mode {MODE}).")

# Pour chaque bulletin, on extrait la liste de ses CVE.
# extraire_cves gère déjà les deux modes : le paramètre d'entrée diffère
# (id de fichier en local, URL en remote).
for b in bulletins_meta:
    param = b["id"] if b["mode"] == "local" else b["lien"]
    b["cves"] = extraire_cves(param, mode=b["mode"], type_bulletin=b["dossier"])

nb_couples = sum(len(b["cves"]) for b in bulletins_meta)
print(f"{nb_couples} couples (bulletin, CVE) au total.")


### 4B. Cache disque autour de get_cve_scores

In [ ]:
# CELLULE 4B — Cache disque autour de get_cve_scores

In [ ]:
# Evite de re-interroger un CVE deja vu (gros gain sur les bulletins a 200+ CVE).
# Persiste en JSON : relancer le notebook ne refait aucun appel API.
import json
import os

CACHE_PATH = "cache_cve.json"
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        cache_cve = json.load(f)
else:
    cache_cve = {}
print(f"Cache initial : {len(cache_cve)} CVE deja connus.")

def get_cve_scores_cached(cve_id):
    """get_cve_scores avec cache disque. Memorise aussi les None (404 MITRE)."""
    if cve_id in cache_cve:
        return cache_cve[cve_id]
    info = get_cve_scores(cve_id)
    cache_cve[cve_id] = info
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache_cve, f, ensure_ascii=False, indent=2)
    return info

### 4C. Enrichissement de chaque CVE unique

In [ ]:
# CELLULE 4C — Enrichissement de chaque CVE unique (une seule fois)

In [ ]:
cve_uniques = sorted({cve for b in bulletins_meta for cve in b["cves"]})
print(f"{len(cve_uniques)} CVE uniques a enrichir.")

for i, cve_id in enumerate(cve_uniques, 1):
    if cve_id in cache_cve:
        continue
    print(f"[{i}/{len(cve_uniques)}] {cve_id}")
    get_cve_scores_cached(cve_id)
print("Enrichissement termine.")

### (Optionnel) Affichage détaillé des CVE
Vérification lisible du contenu enrichi. Passe par le cache, donc rapide après le premier run.

In [ ]:
# =============================================================
# Affichage détaillé des CVE enrichis (via le cache)
# =============================================================
# Cette cellule réutilise le cache rempli par la cellule 4C :
# après le premier enrichissement, elle est quasi instantanée
# (aucun nouvel appel API). On itère sur cve_uniques pour rester
# cohérent avec le mode choisi (remote ou local).

for cve_id in cve_uniques:
    cve_info = get_cve_scores_cached(cve_id)

    if cve_info is None:
        print(f"   [IGNORÉ] {cve_id} absent de MITRE (404).")
        print("-" * 50)
        continue

    print(f"=== Informations pour le {cve_info['identifiant']} ===")
    print(f"Gravité           : {cve_info['base_severity']}")
    print(f"Score CVSS         : {cve_info['score_cvss']}")
    print(f"Score EPSS         : {cve_info['score_epss']}")
    print(f"Code CWE           : {cve_info['type_cwe']}")
    print(f"Produits affectés  : {cve_info['produits_affectes']}")
    print(f"Description        : {cve_info['description']}")
    print("-" * 50)


### 4D. Consolidation en DataFrame (via dataframe.py)

In [ ]:
# CELLULE 4D — Consolidation en DataFrame (via module dataframe.py)
# On reconstruit la structure imbriquee attendue par consolider_en_dataframe :
#   bulletin -> cves_enrichis -> produits_affectes

donnees_globales = []
for b in bulletins_meta:
    donnees_globales.append({
        "id": b["id"],
        "titre": b["titre"],
        "type": b["type"],
        "date_publication": b["date_publication"],
        "lien": b["lien"],
        "cves_enrichis": [info for cve in b["cves"]
                          if (info := cache_cve.get(cve)) is not None],
    })

df = consolider_en_dataframe(donnees_globales, nom_fichier_csv="donnees_enrichies.csv")
print(f"DataFrame : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print("Fichier 'donnees_enrichies.csv' ecrit.")
df.head(10)

## Partie 5 — Interprétation et visualisation

### Chargement et nettoyage
Le CSV stocke les scores en texte avec des marqueurs `N/A`/`Non renseigné`. On convertit en numérique, on ajoute la catégorie de **sévérité** (barème de la consigne) et on crée `df_cve` (un CVE = une ligne).

In [ ]:
# =============================================================
# CELLULE PREP — Chargement + nettoyage des types

In [ ]:
# Le CSV produit par consolider_en_dataframe stocke les scores en texte
# et utilise "N/A" / "Non renseigné" pour les valeurs manquantes.
# On nettoie pour rendre les colonnes numériques exploitables.

import pandas as pd
import numpy as np

# Réutilise df_final si présent, sinon recharge le CSV
try:
    dfv = df.copy()
except NameError:
    dfv = pd.read_csv("donnees_enrichies.csv")

# Les marqueurs de valeur manquante de ton pipeline -> vrais NaN
MANQUANTS = ["N/A", "Non renseigné", "Non disponible", "Non renseigne", ""]
dfv = dfv.replace(MANQUANTS, np.nan)

# Conversion numérique (le CSV garde tout en texte)
dfv["Score CVSS"] = pd.to_numeric(dfv["Score CVSS"], errors="coerce")
dfv["Score EPSS"] = pd.to_numeric(dfv["Score EPSS"], errors="coerce")

# Conversion de la date (format "YYYY-MM-DD" produit par get_bulletin)
dfv["Date de publication"] = pd.to_datetime(dfv["Date de publication"], errors="coerce")

# Ajout de la catégorie de sévérité selon le barème de la consigne
def severite_depuis_cvss(score):
    if pd.isna(score):  return "Non renseigné"
    if score < 4:       return "Faible"      # 0-3
    if score < 7:       return "Moyenne"     # 4-6
    if score < 9:       return "Élevée"      # 7-8
    return "Critique"                        # 9-10

dfv["Sévérité CVSS"] = dfv["Score CVSS"].apply(severite_depuis_cvss)

# Grain "CVE unique" : évite de surcompter un CVE présent sur plusieurs produits
df_cve = dfv.drop_duplicates(subset="Identifiant CVE").copy()

print(f"{dfv.shape[0]} lignes (grain produit), {len(df_cve)} CVE uniques.")
print("\nValeurs manquantes :")
print(dfv[["Score CVSS", "Score EPSS", "Type CWE", "Éditeur (Vendor)"]].isna().sum())
display(dfv[["Score CVSS", "Score EPSS"]].describe())

### G. Histogramme des scores CVSS
Distribution de la gravité, avec repères Élevé (7) et Critique (9).

In [ ]:
# CELLULE G — Histogramme des scores CVSS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

plt.figure()
sns.histplot(df_cve["Score CVSS"].dropna(), bins=20, kde=True, color="#c0392b")
plt.axvline(7, color="orange", ls="--", lw=1, label="Élevé (7)")
plt.axvline(9, color="darkred", ls="--", lw=1, label="Critique (9)")
plt.title("Distribution des scores CVSS")
plt.xlabel("Score CVSS"); plt.ylabel("Nombre de CVE")
plt.legend(); plt.tight_layout(); plt.show()

### H. Répartition par niveau de sévérité

In [ ]:
# CELLULE H — Répartition par niveau de sévérité

In [ ]:
plt.figure(figsize=(7, 7))
ordre = ["Faible", "Moyenne", "Élevée", "Critique", "Non renseigné"]
counts = df_cve["Sévérité CVSS"].value_counts().reindex(ordre).dropna()
couleurs = {"Faible": "#27ae60", "Moyenne": "#f1c40f", "Élevée": "#e67e22",
            "Critique": "#c0392b", "Non renseigné": "#95a5a6"}
plt.pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=90,
        colors=[couleurs[c] for c in counts.index])
plt.title("Répartition des CVE par niveau de sévérité")
plt.tight_layout(); plt.show()

### I. Top 10 des types de faiblesses (CWE)

In [ ]:
# CELLULE I — Top 10 des types de faiblesses (CWE)

In [ ]:
plt.figure()
top_cwe = df_cve["Type CWE"].dropna().value_counts().head(10)
sns.barplot(x=top_cwe.values, y=top_cwe.index, palette="viridis",
            hue=top_cwe.index, legend=False)
plt.title("Top 10 des types de faiblesses (CWE)")
plt.xlabel("Nombre de CVE"); plt.ylabel("Code CWE")
plt.tight_layout(); plt.show()

### J. Distribution des scores EPSS
Échelle log : la majorité des CVE ont une probabilité d'exploitation très faible.

In [ ]:
# CELLULE J — Distribution des scores EPSS

In [ ]:
plt.figure()
sns.histplot(df_cve["Score EPSS"].dropna(), bins=30, color="#2980b9")
plt.title("Distribution des scores EPSS (probabilité d'exploitation)")
plt.xlabel("Score EPSS"); plt.ylabel("Nombre de CVE")
plt.yscale("log")  # la majorité des EPSS sont proches de 0
plt.tight_layout(); plt.show()

### K. Top 15 des éditeurs les plus affectés

In [ ]:
# CELLULE K — Top 15 des éditeurs les plus affectés

In [ ]:
plt.figure(figsize=(10, 6))
top_vendors = dfv["Éditeur (Vendor)"].dropna().value_counts().head(15)
sns.barplot(x=top_vendors.values, y=top_vendors.index, palette="rocket",
            hue=top_vendors.index, legend=False)
plt.title("Top 15 des éditeurs les plus affectés")
plt.xlabel("Nombre de lignes (produit × CVE)"); plt.ylabel("Éditeur")
plt.tight_layout(); plt.show()

### L. Nuage de points CVSS ↔ EPSS
Un CVSS élevé n'implique pas un EPSS élevé : gravité ≠ probabilité d'exploitation.

In [ ]:
# CELLULE L — Nuage de points CVSS vs EPSS

In [ ]:
plt.figure()
sc = df_cve.dropna(subset=["Score CVSS", "Score EPSS"])
sns.scatterplot(data=sc, x="Score CVSS", y="Score EPSS",
                hue="Type", alpha=0.6, s=40)
plt.title("Relation entre gravité (CVSS) et exploitation (EPSS)")
plt.xlabel("Score CVSS"); plt.ylabel("Score EPSS")
plt.tight_layout(); plt.show()

### M. Heatmap de corrélation CVSS / EPSS

In [ ]:
# CELLULE M — Heatmap de corrélation CVSS / EPSS

In [ ]:
plt.figure(figsize=(5, 4))
corr = df_cve[["Score CVSS", "Score EPSS"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", square=True)
plt.title("Corrélation CVSS / EPSS")
plt.tight_layout(); plt.show()
if not corr.isna().all().all():
    print(f"Corrélation de Pearson : {corr.loc['Score CVSS','Score EPSS']:.3f}")

### N. Évolution temporelle (volume + cumul)

In [ ]:
# CELLULE N — Évolution temporelle (volume + cumul)

In [ ]:
temp = df_cve.dropna(subset=["Date de publication"]).copy()
serie = temp.groupby(temp["Date de publication"].dt.date).size().sort_index()
serie_cumul = serie.cumsum()

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(serie.index, serie.values, color="#3498db", alpha=0.6)
ax1.set_ylabel("CVE par jour", color="#3498db")
ax2 = ax1.twinx()
ax2.plot(serie_cumul.index, serie_cumul.values, color="#c0392b", marker="o")
ax2.set_ylabel("Cumul", color="#c0392b")
plt.title("Évolution temporelle des CVE détectées")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

### O. Boxplot des CVSS par éditeur

In [ ]:
# CELLULE O — Boxplot des CVSS par éditeur (top 8)

In [ ]:
plt.figure(figsize=(11, 6))
top8 = dfv["Éditeur (Vendor)"].dropna().value_counts().head(8).index
sub = dfv[dfv["Éditeur (Vendor)"].isin(top8) & dfv["Score CVSS"].notna()]
sns.boxplot(data=sub, x="Score CVSS", y="Éditeur (Vendor)", order=top8,
            palette="Set2", hue="Éditeur (Vendor)", legend=False)
plt.title("Dispersion des scores CVSS pour les 8 éditeurs les plus affectés")
plt.xlabel("Score CVSS"); plt.ylabel("Éditeur")
plt.tight_layout(); plt.show()

### P. Avis vs Alertes : volume et gravité

In [ ]:
# CELLULE P — Avis vs Alertes : volume et gravité

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
df_cve["Type"].value_counts().plot(kind="bar", ax=axes[0], color=["#3498db", "#e74c3c"])
axes[0].set_title("Nombre de CVE par type de bulletin")
axes[0].set_xlabel(""); axes[0].set_ylabel("Nombre de CVE")
axes[0].tick_params(axis="x", rotation=0)

sns.violinplot(data=df_cve.dropna(subset=["Score CVSS"]),
               x="Type", y="Score CVSS", ax=axes[1],
               palette=["#3498db", "#e74c3c"], hue="Type", legend=False)
axes[1].set_title("Distribution des CVSS : Avis vs Alertes")
axes[1].set_xlabel(""); axes[1].set_ylabel("Score CVSS")
plt.tight_layout(); plt.show()